In [1]:
# Install Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-b6pyedjq/unsloth_bcf69dd14278418a9aeaa4461f08aff5
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-b6pyedjq/unsloth_bcf69dd14278418a9aeaa4461f08aff5
  Resolved https://github.com/unslothai/unsloth.git to commit 966d3cda47ef16902ea480fea6c0e590198a8ad7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.6 MB/s eta 0:00:0

In [2]:
# Check GPU
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA Available: True
GPU: Tesla T4


In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None          # Auto detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
)

Unsloth 2026.5.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
from datasets import load_dataset

# Excellent dataset for customer support
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")

def format_prompt(example):
    text = f"""### Instruction:
You are a helpful and professional customer support agent.

### Input:
{example['instruction']}

### Response:
{example['response']}
"""
    return {"text": text}

dataset = dataset.map(format_prompt)
dataset = dataset.shuffle(seed=42).select(range(8000))  # Use 8000 examples for good quality

README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,              # Start with 100 steps (quick test)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/8000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.609097
20,1.088375
30,0.952711
40,0.940921
50,0.875491
60,0.826428
70,0.795134
80,0.794262
90,0.790229
100,0.794708


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.


TrainOutput(global_step=100, training_loss=0.9467355155944824, metrics={'train_runtime': 446.7201, 'train_samples_per_second': 1.791, 'train_steps_per_second': 0.224, 'total_flos': 6755955975782400.0, 'train_loss': 0.9467355155944824, 'epoch': 0.1})

In [7]:
# Merge LoRA weights back into the base model
model = FastLanguageModel.for_inference(model)  # Switch to inference mode

# Save locally first
model.save_pretrained("llama3.1-customer-support-finetuned")
tokenizer.save_pretrained("llama3.1-customer-support-finetuned")

Unsloth: Restored added_tokens_decoder metadata in llama3.1-customer-support-finetuned/tokenizer_config.json.


('llama3.1-customer-support-finetuned/tokenizer_config.json',
 'llama3.1-customer-support-finetuned/chat_template.jinja',
 'llama3.1-customer-support-finetuned/tokenizer.json')

In [8]:
# Login to Hugging Face
from huggingface_hub import login
login()   # Paste your Hugging Face token when prompted

In [9]:
# Push model to Hugging Face
model.push_to_hub("victor-tu-customer-support-llama3.1-8b", private=False)
tokenizer.push_to_hub("victor-tu-customer-support-llama3.1-8b")

README.md:   0%|          | 0.00/577 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  557kB /  168MB            

Saved model to https://huggingface.co/victor-tu-customer-support-llama3.1-8b


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpfz3w2tjj/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpfz3w2tjj/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

In [10]:
# Test it
messages = [
    {"role": "user", "content": "My order hasn't arrived yet. It's been 10 days. Can you help me?"}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

My order hasn't arrived yet. It's been 10 days. Can you help me?assistant

I'm sorry to hear that your order hasn't arrived yet and it has been 10 days. I'm here to assist you and help resolve this issue. To better understand the situation, could you please provide me with the order number or any other relevant details? This will allow me to investigate and provide you with the necessary support.


In [12]:
import gradio as gr

def customer_support_chat(message, history):
    messages = []
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean up response (remove the prompt part)
    if "### Response:" in response:
        response = response.split("### Response:")[-1].strip()

    return response

# Create Gradio Interface
demo = gr.ChatInterface(
    fn=customer_support_chat,
    title="🤖 Victor's Customer Support AI Assistant",
    description="Fine-tuned Llama 3.1 8B for professional customer support help.",
    examples=[
        ["My package is delayed by 5 days. What can I do?"],
        ["I want to return an item. What's the process?"],
        ["Can you track my order #ORD123456?"]
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c6020208e7033fb1d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
